# Toy-ParaRNN MVP: parity task через систему нелинейных уравнений

Цель ноутбука — показать на простой toy-RNN, что рекуррентное вычисление можно:

1. выполнить обычным последовательным проходом;
2. переписать как систему уравнений `F(h)=0`;
3. решить методом Ньютона;
4. на шаге Ньютона получить линейную рекурсию;
5. решить эту линейную рекурсию через toy parallel scan / doubling.

Это учебная демонстрация идеи, а не реализация настоящей Apple ParaRNN.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import math

np.random.seed(42)

ALPHA = 2.0
H0 = 1.0


## 1. Parity task и данные

В parity task входом является бинарная последовательность `x_t ∈ {0, 1}`.
Метка равна `0`, если число единиц чётное, и `1`, если число единиц нечётное.

Для toy-RNN удобно заменить вход на знак:

```text
s_t = 1 - 2x_t

x_t = 0 -> s_t = +1
x_t = 1 -> s_t = -1
```


In [ ]:
def parity_label(x: np.ndarray) -> int:
    """
    Возвращает 0, если количество единиц в x чётное.
    Возвращает 1, если количество единиц в x нечётное.
    """
    return int(np.sum(x) % 2)


def generate_sequence(seq_len: int, seed: int | None = None) -> np.ndarray:
    """
    Генерирует одну бинарную последовательность длины seq_len.
    Возвращает numpy array формы (seq_len,).
    """
    if seed is not None:
        rng = np.random.default_rng(seed)
        return rng.integers(0, 2, size=seq_len, dtype=int)

    return np.random.randint(0, 2, size=seq_len)


def generate_batch(
    batch_size: int,
    seq_len: int,
    seed: int | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Генерирует batch бинарных последовательностей.

    Возвращает:
    X: shape (batch_size, seq_len)
    y: shape (batch_size,)
    """
    rng = np.random.default_rng(seed)
    X = rng.integers(0, 2, size=(batch_size, seq_len), dtype=int)
    y = np.array([parity_label(row) for row in X], dtype=int)
    return X, y


def s_from_x(x: np.ndarray) -> np.ndarray:
    """
    Преобразует x из {0, 1} в s из {+1, -1}.
    s = 1 - 2x
    """
    x = np.asarray(x, dtype=int)
    return 1 - 2 * x


In [ ]:
assert parity_label(np.array([0, 0, 0])) == 0
assert parity_label(np.array([1, 0, 0])) == 1
assert parity_label(np.array([1, 1, 0])) == 0
assert np.array_equal(s_from_x(np.array([0, 1, 0, 1])), np.array([1, -1, 1, -1]))

X_check, y_check = generate_batch(batch_size=4, seq_len=6, seed=0)
assert X_check.shape == (4, 6)
assert y_check.shape == (4,)
assert all(y_check[i] == parity_label(X_check[i]) for i in range(len(y_check)))

print("Data helpers: OK")


## 2. Обычная последовательная toy-RNN

Скрытое состояние кодирует чётность префикса знаком:

```text
h_t > 0 -> чётная чётность префикса
h_t < 0 -> нечётная чётность префикса
```

Начинаем с `h_0 = +1`. Рекуррентное правило:

```text
h_t = tanh(alpha * s_t * h_{t-1}), alpha = 2.0
```


In [ ]:
def rnn_step(h_prev: float, x_t: int, alpha: float = ALPHA) -> float:
    """
    Один шаг toy-RNN:
    h_t = tanh(alpha * s_t * h_{t-1})
    """
    s_t = 1 - 2 * int(x_t)
    return float(np.tanh(alpha * s_t * h_prev))


def sequential_rnn(
    x: np.ndarray,
    alpha: float = ALPHA,
    h0: float = H0,
) -> np.ndarray:
    """
    Обычный последовательный проход toy-RNN.

    Вход:
    x: массив из 0 и 1 длины L.

    Выход:
    h: массив длины L+1:
       h[0] = h0
       h[1] = h_1
       ...
       h[L] = h_L
    """
    x = np.asarray(x)
    h = np.zeros(len(x) + 1)
    h[0] = h0

    for t in range(len(x)):
        h[t + 1] = rnn_step(h[t], x[t], alpha)

    return h


def predict_from_hidden(h_last: float) -> int:
    """
    Если h_last > 0, возвращает 0.
    Если h_last < 0, возвращает 1.
    """
    return int(h_last < 0)


In [ ]:
examples = [
    np.array([0, 1, 0, 1]),
    np.array([1, 0, 1, 1]),
    np.array([0, 0, 0, 0]),
    np.array([1, 1, 1, 1]),
    np.array([1, 0, 0, 0]),
]

for x in examples:
    h = sequential_rnn(x)
    y_true = parity_label(x)
    y_pred = predict_from_hidden(h[-1])
    print("x =", x, "true =", y_true, "pred =", y_pred, "h_last =", h[-1])
    assert y_true == y_pred


## 3. Hidden state как память о префиксе

На каждом шаге знак `h_t` должен совпадать с parity уже прочитанного префикса.
Для `t=0` префикс пустой, поэтому parity равна `0`.


In [ ]:
def prefix_parities(x: np.ndarray) -> np.ndarray:
    """
    Возвращает массив длины L+1.
    prefix_parities[t] = parity для префикса x[:t].
    Для t=0 parity равна 0, потому что единиц пока нет.
    """
    out = [0]
    current = 0

    for value in x:
        current = (current + int(value)) % 2
        out.append(current)

    return np.array(out)


x_demo = np.array([1, 0, 1, 1])
h_demo = sequential_rnn(x_demo)
p_demo = prefix_parities(x_demo)

for t in range(len(h_demo)):
    x_text = "-" if t == 0 else str(x_demo[t - 1])
    sign_text = "even" if h_demo[t] > 0 else "odd"
    print(
        f"t={t}, x={x_text}, h={h_demo[t]:+.4f}, "
        f"sign={sign_text}, prefix_parity={p_demo[t]}"
    )

assert len(p_demo) == len(h_demo)
assert all((h_demo[t] > 0) == (p_demo[t] == 0) for t in range(len(h_demo)))


In [ ]:
def plot_hidden_states(
    x: np.ndarray,
    h: np.ndarray,
    title: str = "Hidden states",
) -> None:
    """
    Строит график h_t по времени.
    Также рисует горизонтальную линию y=0.
    """
    plt.figure(figsize=(8, 4))
    plt.plot(range(len(h)), h, marker="o")
    plt.axhline(0, linestyle="--")
    plt.xlabel("t")
    plt.ylabel("h_t")
    plt.title(title)
    plt.grid(True)
    plt.show()


plot_hidden_states(x_demo, h_demo, title="Toy-RNN hidden states for x = [1, 0, 1, 1]")


## 4. Переписываем RNN как систему уравнений `F(h)=0`

Обычная RNN задаёт уравнения:

```text
h_t = tanh(alpha * s_t * h_{t-1})
```

Переносим всё в левую часть:

```text
F_t(h) = h_t - tanh(alpha * s_t * h_{t-1}) = 0
```

Теперь неизвестными являются сразу все состояния:

```text
h = [h_1, h_2, ..., h_L]
```

Если `F(h)=0`, значит все рекуррентные правила выполнены одновременно.


In [ ]:
def residual_F(
    h_unknown: np.ndarray,
    x: np.ndarray,
    alpha: float = ALPHA,
    h0: float = H0,
) -> np.ndarray:
    """
    Считает F(h) для системы уравнений toy-RNN.

    h_unknown: массив длины L, содержащий [h_1, ..., h_L]
    x: бинарная последовательность длины L

    Возвращает:
    F: массив длины L, где
       F[t] = h_t - tanh(alpha * s_t * h_{t-1})

    Важно:
    в коде индексация с нуля:
       h_unknown[0] соответствует h_1
       h_unknown[t] соответствует h_{t+1}
    """
    x = np.asarray(x)
    h_unknown = np.asarray(h_unknown)
    L = len(x)
    s = s_from_x(x)

    F = np.zeros(L)

    for t in range(L):
        h_t = h_unknown[t]
        if t == 0:
            h_prev = h0
        else:
            h_prev = h_unknown[t - 1]

        F[t] = h_t - np.tanh(alpha * s[t] * h_prev)

    return F


def residual_norm(
    h_unknown: np.ndarray,
    x: np.ndarray,
    alpha: float = ALPHA,
    h0: float = H0,
) -> float:
    """
    Возвращает ||F(h)||_2.
    """
    return float(np.linalg.norm(residual_F(h_unknown, x, alpha, h0)))


In [ ]:
x = np.array([1, 0, 1, 1])
h_seq = sequential_rnn(x)

F_at_seq = residual_F(h_seq[1:], x)
print("F(h_seq) =", F_at_seq)
print("||F(h_seq)|| =", np.linalg.norm(F_at_seq))

assert np.linalg.norm(F_at_seq) < 1e-12


## 5. Якобиан и Newton step

Метод Ньютона решает систему `F(h)=0` итерациями.

На каждой итерации есть текущее приближение `h`.
Мы ищем поправку `delta`:

```text
h_new = h + delta
```

Поправка находится из линейной системы:

```text
J delta = -F(h)
```

`J` — это матрица производных. В нашем случае она простая:
каждое уравнение `F_t` зависит только от `h_t` и `h_{t-1}`.
Поэтому матрица почти пустая: на диагонали стоят `1`,
а под диагональю стоят коэффициенты, связанные с производной `tanh`.

У нас:

```text
F_t = h_t - tanh(alpha * s_t * h_{t-1})
```

Производная по предыдущему состоянию:

```text
d/dh_{t-1} tanh(alpha * s_t * h_{t-1})
= alpha * s_t * (1 - tanh(alpha * s_t * h_{t-1})^2)
```

Обозначим:

```text
a_t = alpha * s_t * (1 - tanh(alpha * s_t * h_{t-1})^2)
```

Тогда Newton step превращается в линейную рекурсию:

```text
delta_t = a_t * delta_{t-1} + b_t
```

где:

```text
b_t = -F_t
```

В коде индексация с нуля:

```text
delta[i] соответствует поправке для h_{i+1}
```

Для первого уравнения `h_0` фиксирован, поэтому:

```text
a[0] = 0
```


In [ ]:
def jacobian_coefficients(
    h_unknown: np.ndarray,
    x: np.ndarray,
    alpha: float = ALPHA,
    h0: float = H0,
) -> np.ndarray:
    """
    Возвращает массив a длины L для линейной рекурсии Newton step:

        delta[i] = a[i] * delta[i-1] + b[i]

    Для i=0:
        a[0] = 0

    Для i>=1:
        a[i] = alpha * s[i] * (1 - tanh(alpha * s[i] * h_unknown[i-1])**2)
    """
    x = np.asarray(x)
    h_unknown = np.asarray(h_unknown)
    L = len(x)
    s = s_from_x(x)

    a = np.zeros(L)

    for i in range(1, L):
        z = alpha * s[i] * h_unknown[i - 1]
        a[i] = alpha * s[i] * (1.0 - np.tanh(z) ** 2)

    return a


def build_dense_jacobian(a: np.ndarray) -> np.ndarray:
    """
    Строит плотную матрицу J размера L x L.

    На диагонали стоят 1.
    Под диагональю стоят -a[i].

    J delta = -F
    """
    a = np.asarray(a)
    L = len(a)
    J = np.eye(L)

    for i in range(1, L):
        J[i, i - 1] = -a[i]

    return J


## 6. Решение линейной рекурсии для Newton step

Сначала реализуем обычное последовательное решение:

```text
delta[i] = a[i] * delta[i-1] + b[i]
```

Считаем, что `delta[-1] = 0`, поэтому `delta[0] = b[0]`.


In [ ]:
def solve_linear_recurrence_sequential(
    a: np.ndarray,
    b: np.ndarray,
) -> np.ndarray:
    """
    Решает линейную рекурсию:

        delta[i] = a[i] * delta[i-1] + b[i]

    Считаем, что delta[-1] = 0.
    Поэтому для i=0:
        delta[0] = b[0]
    """
    a = np.asarray(a)
    b = np.asarray(b)
    L = len(a)

    delta = np.zeros(L)
    prev = 0.0

    for i in range(L):
        delta[i] = a[i] * prev + b[i]
        prev = delta[i]

    return delta


## 7. Toy parallel scan / doubling

Линейную рекурсию можно представить как последовательность преобразований:

```text
T_i(z) = a_i * z + b_i
```

Тогда:

```text
delta_0 = T_0(0)
delta_1 = T_1(T_0(0))
delta_2 = T_2(T_1(T_0(0)))
```

Нужно посчитать префиксные композиции преобразований.

Композиция двух преобразований снова имеет такой же вид:

```text
если
T_left(z) = a1 * z + b1
T_right(z) = a2 * z + b2

то
T_right(T_left(z)) = (a2*a1) * z + (a2*b1 + b2)
```


In [ ]:
def compose_affine(
    pair_right: tuple[float, float],
    pair_left: tuple[float, float],
) -> tuple[float, float]:
    """
    Композиция двух affine-преобразований.

    pair_left = (a1, b1), задаёт T_left(z) = a1*z + b1
    pair_right = (a2, b2), задаёт T_right(z) = a2*z + b2

    Возвращает параметры T_right(T_left(z)).
    """
    a2, b2 = pair_right
    a1, b1 = pair_left
    return a2 * a1, a2 * b1 + b2


def solve_linear_recurrence_scan(
    a: np.ndarray,
    b: np.ndarray,
) -> np.ndarray:
    """
    Демонстрационное решение рекурсии:

        delta[i] = a[i] * delta[i-1] + b[i]

    через идею parallel prefix / doubling.

    Важно:
    это не настоящая параллельная CUDA-реализация.
    На CPU это может быть медленнее обычного цикла.
    Цель — показать идею уменьшения глубины зависимостей.
    """
    a = np.asarray(a).copy()
    b = np.asarray(b).copy()
    L = len(a)

    A = a.copy()
    B = b.copy()

    step = 1
    while step < L:
        A_old = A.copy()
        B_old = B.copy()

        for i in range(step, L):
            # Compose current prefix transform with the prefix step positions before it.
            A[i] = A_old[i] * A_old[i - step]
            B[i] = A_old[i] * B_old[i - step] + B_old[i]

        step *= 2

    # delta[-1] = 0, therefore the result is B.
    return B


In [ ]:
assert compose_affine((3.0, 4.0), (2.0, 1.0)) == (6.0, 7.0)

x_test = generate_sequence(seq_len=8, seed=1)
h_current = np.random.randn(len(x_test)) * 0.1

F = residual_F(h_current, x_test)
a = jacobian_coefficients(h_current, x_test)
b = -F

delta_seq = solve_linear_recurrence_sequential(a, b)
delta_scan = solve_linear_recurrence_scan(a, b)

J = build_dense_jacobian(a)
delta_dense = np.linalg.solve(J, b)

print("max |seq - scan| =", np.max(np.abs(delta_seq - delta_scan)))
print("max |seq - dense| =", np.max(np.abs(delta_seq - delta_dense)))

assert np.max(np.abs(delta_seq - delta_scan)) < 1e-10
assert np.max(np.abs(delta_seq - delta_dense)) < 1e-10


## 8. Newton solver

Для устойчивого MVP мы используем `sign initialization`.
Она задаёт правильный знак состояния, но не точное значение `h_t`.
Это нужно, чтобы Newton solver стабильно сходился на длинных последовательностях.

Отдельно ниже показано, что на коротких последовательностях старт из нулей тоже работает.


In [ ]:
def initial_guess(
    x: np.ndarray,
    mode: str = "sign",
    magnitude: float = 0.9,
    h0: float = H0,
) -> np.ndarray:
    """
    Возвращает начальное приближение для [h_1, ..., h_L].

    mode="zeros":
        все нули.

    mode="sign":
        используем правильный знак parity-префикса, но не точную величину.
        h_init[t] = magnitude * h0 * cumulative_product(s[:t+1])

    Для MVP mode="sign" используется как стабильный старт для Newton,
    особенно на длинных последовательностях.
    """
    x = np.asarray(x)
    L = len(x)

    if mode == "zeros":
        return np.zeros(L)

    if mode == "sign":
        s = s_from_x(x)
        signs = h0 * np.cumprod(s)
        return magnitude * signs

    raise ValueError(f"Unknown init mode: {mode}")


def newton_solver(
    x: np.ndarray,
    alpha: float = ALPHA,
    h0: float = H0,
    max_iter: int = 20,
    tol: float = 1e-10,
    init_mode: str = "sign",
    init_magnitude: float = 0.9,
    linear_solver: str = "sequential",
    verbose: bool = False,
) -> tuple[np.ndarray, list[float]]:
    """
    Решает систему F(h)=0 методом Ньютона.

    Возвращает:
    h_unknown: найденные состояния [h_1, ..., h_L]
    history: список значений ||F(h)|| на каждой итерации
    """
    h = initial_guess(x, mode=init_mode, magnitude=init_magnitude, h0=h0)
    history = []

    for k in range(max_iter):
        F = residual_F(h, x, alpha=alpha, h0=h0)
        norm = np.linalg.norm(F)
        history.append(float(norm))

        if verbose:
            print(f"iter={k}, residual={norm:.3e}")

        if norm < tol:
            break

        a = jacobian_coefficients(h, x, alpha=alpha, h0=h0)
        b = -F

        if linear_solver == "sequential":
            delta = solve_linear_recurrence_sequential(a, b)
        elif linear_solver == "scan":
            delta = solve_linear_recurrence_scan(a, b)
        else:
            raise ValueError(f"Unknown linear_solver: {linear_solver}")

        h = h + delta

    return h, history


In [ ]:
x = generate_sequence(seq_len=32, seed=123)

h_seq_full = sequential_rnn(x)
h_seq = h_seq_full[1:]

h_newton_seq, hist_seq = newton_solver(
    x,
    init_mode="sign",
    linear_solver="sequential",
    verbose=True,
)

h_newton_scan, hist_scan = newton_solver(
    x,
    init_mode="sign",
    linear_solver="scan",
    verbose=True,
)

print("Sequential RNN final h:", h_seq[-1])
print("Newton sequential final h:", h_newton_seq[-1])
print("Newton scan final h:", h_newton_scan[-1])

print("max |RNN - Newton sequential| =", np.max(np.abs(h_seq - h_newton_seq)))
print("max |RNN - Newton scan| =", np.max(np.abs(h_seq - h_newton_scan)))
print("max |Newton seq - Newton scan| =", np.max(np.abs(h_newton_seq - h_newton_scan)))

assert np.max(np.abs(h_seq - h_newton_seq)) < 1e-8
assert np.max(np.abs(h_seq - h_newton_scan)) < 1e-8
assert np.max(np.abs(h_newton_seq - h_newton_scan)) < 1e-10

y_true = parity_label(x)
y_rnn = predict_from_hidden(h_seq[-1])
y_newton = predict_from_hidden(h_newton_seq[-1])
y_scan = predict_from_hidden(h_newton_scan[-1])

print("true:", y_true)
print("rnn:", y_rnn)
print("newton:", y_newton)
print("newton+scan:", y_scan)

assert y_true == y_rnn == y_newton == y_scan


## 9. График сходимости Ньютона

Сравним норму residual для двух вариантов решения линейной рекурсии:
обычный последовательный solve и toy scan/doubling.


In [ ]:
def plot_residual_history(
    histories: dict,
    title: str = "Newton residual norm",
) -> None:
    """
    histories: dict name -> list of residual norms
    """
    plt.figure(figsize=(8, 4))

    for name, history in histories.items():
        values = np.maximum(np.asarray(history, dtype=float), 1e-16)
        plt.plot(range(len(values)), values, marker="o", label=name)

    plt.yscale("log")
    plt.xlabel("Newton iteration")
    plt.ylabel("||F(h)||")
    plt.title(title)
    plt.grid(True)
    plt.legend()
    plt.show()


plot_residual_history(
    {
        "Newton + sequential solve": hist_seq,
        "Newton + scan solve": hist_scan,
    },
    title="Residual decay for Newton solver",
)


## 10. Сравнение методов на batch

Проверим, что все три метода дают accuracy `1.0` на 100 случайных примерах длины 64:

1. обычная sequential RNN;
2. Newton + sequential linear solve;
3. Newton + scan linear solve.


In [ ]:
def evaluate_methods(
    num_examples: int = 100,
    seq_len: int = 32,
    seed: int = 0,
) -> dict:
    """
    Генерирует num_examples последовательностей.
    Сравнивает accuracy для:
    1. sequential RNN
    2. Newton + sequential linear solve
    3. Newton + scan linear solve

    Возвращает словарь с accuracy.
    """
    X, y = generate_batch(num_examples, seq_len, seed=seed)

    correct_rnn = 0
    correct_newton_seq = 0
    correct_newton_scan = 0

    for i in range(num_examples):
        x = X[i]
        true = y[i]

        h_rnn = sequential_rnn(x)
        pred_rnn = predict_from_hidden(h_rnn[-1])

        h_nseq, _ = newton_solver(x, linear_solver="sequential", init_mode="sign")
        pred_nseq = predict_from_hidden(h_nseq[-1])

        h_nscan, _ = newton_solver(x, linear_solver="scan", init_mode="sign")
        pred_nscan = predict_from_hidden(h_nscan[-1])

        correct_rnn += int(pred_rnn == true)
        correct_newton_seq += int(pred_nseq == true)
        correct_newton_scan += int(pred_nscan == true)

    return {
        "sequential_rnn": correct_rnn / num_examples,
        "newton_sequential_solve": correct_newton_seq / num_examples,
        "newton_scan_solve": correct_newton_scan / num_examples,
    }


acc = evaluate_methods(num_examples=100, seq_len=64, seed=42)
print(acc)

assert acc["sequential_rnn"] == 1.0
assert acc["newton_sequential_solve"] == 1.0
assert acc["newton_scan_solve"] == 1.0


## 11. Dependency depth: `L` против `log2(L)`

Обычная RNN имеет цепочку зависимостей длины `L`.
Toy scan / doubling имеет примерно `log2(L)` уровней объединения.

Это не значит, что Python-код будет быстрее.
Это значит, что теоретическая глубина последовательных зависимостей меньше.


In [ ]:
def plot_dependency_depth(lengths: list[int]) -> None:
    rnn_depth = np.array(lengths)
    scan_depth = np.array([math.ceil(math.log2(L)) if L > 1 else 1 for L in lengths])

    plt.figure(figsize=(8, 4))
    plt.plot(lengths, rnn_depth, marker="o", label="Sequential RNN depth = L")
    plt.plot(lengths, scan_depth, marker="o", label="Scan/doubling depth ≈ log2(L)")
    plt.xlabel("Sequence length L")
    plt.ylabel("Dependency depth")
    plt.title("Sequential dependency depth vs scan/doubling depth")
    plt.grid(True)
    plt.legend()
    plt.show()


lengths = [4, 8, 16, 32, 64, 128, 256, 512]
plot_dependency_depth(lengths)


## 12. Мини-бенчмарк времени на CPU

Цель бенчмарка — не доказать ускорение, а честно показать время выполнения в Python на CPU.


In [ ]:
def time_function(fn, repeats: int = 5) -> float:
    """
    Возвращает среднее время выполнения fn() по repeats запускам.
    """
    times = []

    for _ in range(repeats):
        start = time.perf_counter()
        fn()
        end = time.perf_counter()
        times.append(end - start)

    return float(np.mean(times))


def benchmark_methods(
    lengths: list[int],
    repeats: int = 3,
    seed: int = 0,
) -> dict:
    """
    Для каждой длины L сравнивает время:
    1. sequential_rnn
    2. newton_solver with sequential linear solve
    3. newton_solver with scan linear solve

    Возвращает словарь результатов.
    """
    results = {
        "L": [],
        "sequential_rnn": [],
        "newton_seq": [],
        "newton_scan": [],
    }

    for L in lengths:
        x = generate_sequence(L, seed=seed + L)

        t_rnn = time_function(lambda: sequential_rnn(x), repeats=repeats)

        t_newton_seq = time_function(
            lambda: newton_solver(
                x,
                init_mode="sign",
                linear_solver="sequential",
                max_iter=20,
                tol=1e-10,
            ),
            repeats=repeats,
        )

        t_newton_scan = time_function(
            lambda: newton_solver(
                x,
                init_mode="sign",
                linear_solver="scan",
                max_iter=20,
                tol=1e-10,
            ),
            repeats=repeats,
        )

        results["L"].append(L)
        results["sequential_rnn"].append(t_rnn)
        results["newton_seq"].append(t_newton_seq)
        results["newton_scan"].append(t_newton_scan)

        print(
            f"L={L:4d} | "
            f"RNN={t_rnn:.6f}s | "
            f"NewtonSeq={t_newton_seq:.6f}s | "
            f"NewtonScan={t_newton_scan:.6f}s"
        )

    return results


def plot_benchmark_results(results: dict) -> None:
    L = results["L"]

    plt.figure(figsize=(8, 4))
    plt.plot(L, results["sequential_rnn"], marker="o", label="Sequential RNN")
    plt.plot(L, results["newton_seq"], marker="o", label="Newton + sequential solve")
    plt.plot(L, results["newton_scan"], marker="o", label="Newton + scan solve")
    plt.xlabel("Sequence length L")
    plt.ylabel("Average runtime, seconds")
    plt.title("Runtime comparison on CPU")
    plt.grid(True)
    plt.legend()
    plt.show()


lengths_bench = [8, 16, 32, 64, 128, 256]
bench_results = benchmark_methods(lengths_bench, repeats=3, seed=123)
plot_benchmark_results(bench_results)


На обычном CPU и в Python toy scan не обязан быть быстрее обычной RNN.
Это нормально: наша цель — показать структуру вычислений, а не получить реальное ускорение.
Настоящее ускорение требует эффективной параллельной реализации, например на GPU.


## 13. Дополнительный эксперимент: Newton из нулей

Покажем короткий пример для `L=16`: Newton может стартовать не только с sign initialization,
но на больших `L` такой старт может быть менее стабильным.


In [ ]:
x_small = generate_sequence(seq_len=16, seed=7)

h_zero_init, hist_zero = newton_solver(
    x_small,
    init_mode="zeros",
    linear_solver="sequential",
    max_iter=20,
    tol=1e-10,
    verbose=True,
)

h_sign_init, hist_sign = newton_solver(
    x_small,
    init_mode="sign",
    linear_solver="sequential",
    max_iter=20,
    tol=1e-10,
    verbose=True,
)

plot_residual_history(
    {
        "Newton from zeros": hist_zero,
        "Newton from sign init": hist_sign,
    },
    title="Newton initialization comparison, L=16",
)

h_small_seq = sequential_rnn(x_small)[1:]
assert np.max(np.abs(h_zero_init - h_small_seq)) < 1e-8
assert np.max(np.abs(h_sign_init - h_small_seq)) < 1e-8


Старт из нулей возможен на коротких последовательностях,
но может давать большие промежуточные поправки.
Для стабильного MVP на более длинных `L` используем sign initialization.


## Выводы MVP

В этом ноутбуке мы реализовали toy-RNN для parity task.

1. Обычная RNN правильно решает parity task, двигаясь слева направо.
2. Скрытое состояние `h_t` хранит чётность уже прочитанного префикса.
3. Рекуррентные правила можно переписать как систему уравнений `F(h)=0`.
4. Последовательный проход RNN действительно даёт решение этой системы: `F(h)≈0`.
5. Метод Ньютона находит те же скрытые состояния.
6. На шаге Ньютона возникает линейная рекурсия для поправок `delta`.
7. Эту линейную рекурсию можно решить обычным циклом или toy scan/doubling.
8. Scan/doubling показывает идею уменьшения глубины зависимостей с `L` до примерно `log2(L)`.
9. Python-реализация на CPU не обязана быть быстрее — это учебная демонстрация, а не высокопроизводительная реализация.
10. Проект связан с идеей ParaRNN концептуально, но не является реализацией Apple ParaRNN.
